In [1]:
import pandas as pd
data_train = pd.read_parquet(r"final_data_http\chunk-based-split-3\train_df_prepared.parquet")
data_valid = pd.read_parquet(r"final_data_http\chunk-based-split-3\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"final_data_http\chunk-based-split-3\test_df_prepared.parquet")

In [2]:
import pandas as pd
import numpy as np
import torch
import gc
from torch_geometric.data import Data # thư viên Data giúp lưu trữ đồ thị lưới 
from tqdm.notebook import tqdm
import os

# bật cảnh báo trùng lặp của KMP (Intel MKL) để tránh lỗi crash khi dùng nhiều luồng trên CPU
os.environ["KMP_DUPLICATE_BLOCK_WARNING"] = "TRUE"

# Xây dựng đồ thị Flow-based Graph có đặc trưng cạnh
def build_ip_topology_graph(df, train_mask, valid_mask, test_mask, window_size=10):
    print(f"Bước 1: Trích xuất Node Features (X) và Label (y)...")
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    
    # các cột cần drop 
    cols_present = [c for c in cols_to_drop if c in df.columns]

    # các cột còn lại sẽ là feature
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    # tạo một numpy array chứa các feature
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    
    # ascontiguousarray: đảm bảo dữ liệu được lưu trữ liên tục trong bộ nhớ
    features_np = np.ascontiguousarray(features_np)
    
    # lấy node features (x_tensor) và label (y_tensor)
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()



    print(f"Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...")

    # lấy các giá trị để xây dựng cạnh
    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    # tạo một mảng split_id để đánh dấu tập Train (0), Valid (1), Test (2)
    split_id = np.zeros(len(df), dtype=np.int8)
    split_id[valid_mask.numpy()] = 1
    split_id[test_mask.numpy()] = 2
    

    # gom nhóm các gói tin theo ip để nối cạnh
    ip_to_indices = {}
    for idx in tqdm(range(len(df)), desc="[1/2] Gom nhóm Gói tin theo IP"):
        s_ip = src_ips[idx]
        d_ip = dst_ips[idx]
        
        # ip_to_indices: dictionary mapping mỗi IP (cả src và dst) đến danh sách các chỉ số của gói tin liên quan đến IP đó
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx:
            ip_to_indices[s_ip].append(idx)
            
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx:
            ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    
    # nối cạnh giữa các gói tin có cùng IP (src hoặc dst) trong một cửa sổ thời gian nhất định (window_size)
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Căng lưới đồ thị (Window = {window_size})"):
        n_flows = len(indices)
        if n_flows < 2: continue
        
        for i in range(1, n_flows):
            curr_idx = indices[i]
            start_w = max(0, i - window_size) # start_w: chỉ số bắt đầu của cửa sổ (window) để tìm các gói tin trước đó có cùng IP
            
            # duyệt qua các gói tin trước đó trong cửa sổ để tạo cạnh
            for j in range(start_w, i):
                past_idx = indices[j]
                
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx]) # tính khoảng thời gian giữa 2 gói tin
                
                # tuy nhiên, nếu 2 gói tin cách nhau hơn 60s thì không nói cạnh 
                if dt_raw > 60.0:
                    continue
                
                # tính toán các đặc trưng cạnh
                # chênh lệch thời gian: (Log scale + Min-Max Scaling về dải [0, 1])
                # giá trị max lý thuyết khi dt_raw = 60s là log1p(60,000,000) ~ 17.9 => chia cho 18
                dt = np.log1p(dt_raw * 1e6) / 18.0
                # binary liệu 2 gói tin này có chung địa chỉ IP nguồn không (cùng người gửi với nhau)
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                # liệu 2 gói tin này có cùng cổng nguồn không?
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                # Liệu 2 gói tin này có cùng cổng đích không?
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0

                # list lưu các đặc trưng cạnh
                attr = [dt, same_dir, same_sport, same_dport]
                
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr) 
                    
    print("\nHợp nhất và tạo Cạnh (Culling Edge Index)...")
    # loại bỏ các cạnh trùng lặp (nếu có) và tạo edge_index, edge_attr
    
    # tạo dataframe tạm thời để bỏ cạnh trùng lặp
    df_edges = pd.DataFrame({
        'src': all_src, 
        'dst': all_dst,
        'attr0': [a[0] for a in all_edge_attrs],
        'attr1': [a[1] for a in all_edge_attrs],
        'attr2': [a[2] for a in all_edge_attrs],
        'attr3': [a[3] for a in all_edge_attrs]
    })
    
    # loại bỏ các cạnh trùng lặp (nếu có) dựa trên cột 'src' và 'dst'
    df_edges = df_edges.drop_duplicates(subset=['src', 'dst']).reset_index(drop=True)
    
    # edge_index: tensor 2 x num_edges, edge_attr: tensor num_edges x num_edge_features
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['attr0', 'attr1', 'attr2', 'attr3']].values, dtype=torch.float32).contiguous()
    
    print(f"HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!")
    print(f"   - Số Nodes : {x_tensor.size(0):,}")
    print(f"   - Số Edges : {edge_index.size(1):,}")
    print(f"   - Số Features/Cạnh : {edge_attr.shape[1]}")
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    
    # tạo đồ thị PyG Data với node features, edge index, edge attributes và label
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    
    # gắn mask cho tập train, valid, test vào đồ thị
    graph.train_mask = train_mask
    graph.valid_mask = valid_mask
    graph.test_mask = test_mask
    return graph

In [ ]:
import gc
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from torch_geometric.loader import NeighborLoader
import torch
import numpy as np
import pandas as pd

print("=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===")

# tạo 1 cột để đánh dấu tập dữ liệu (train/valid/test) trước khi gộp để tránh Data Leakage
print("Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...")
data_train['split_tag'] = 0
data_valid['split_tag'] = 1
data_test['split_tag'] = 2

# concat lại thành một dataframe chuẩn
df_all = pd.concat([data_train, data_valid, data_test], ignore_index=True)

# xóa các dataframe gốc để giải phóng bộ nhớ
del data_train, data_valid, data_test
gc.collect()

# chuyển datetime sang số để sắp xếp và giữ lại micro giây
df_all['timestamp_num'] = pd.to_datetime(df_all['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

print("Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...")
# Sắp xếp tĩnh trực tiếp (INPLACE) để chặn Pandas tạo bản copy 9GB trong bộ nhớ RAM
df_all.sort_values(by='timestamp_num', inplace=True)
df_all.reset_index(drop=True, inplace=True)
gc.collect()

total_nodes = len(df_all)

# tạo mask cho đồ thị
print("Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...")
train_mask = torch.tensor((df_all['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((df_all['split_tag'] == 1).values, dtype=torch.bool)
test_mask  = torch.tensor((df_all['split_tag'] == 2).values, dtype=torch.bool)

# xóa cột split_tag
df_all.drop(columns=['split_tag'], inplace=True)
gc.collect()

# in ra thông tin thống kê về số lượng node và phân bố mask
print(f" - Tổng Node: {total_nodes:,}")
print(f" - Train Mask : {train_mask.sum().item():,} ({train_mask.float().mean()*100:.1f}%)")
print(f" - Valid Mask : {valid_mask.sum().item():,} ({valid_mask.float().mean()*100:.1f}%)")
print(f" - Test Mask  : {test_mask.sum().item():,} ({test_mask.float().mean()*100:.1f}%)")

print("Hoàn tất Data Prep cho Masks và Sorting Thời gian.")

# build đồ thị với đặc trưng cạnh và gắn mask
print("\n=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===")
super_graph_faiss = build_ip_topology_graph(df_all, train_mask, valid_mask, test_mask, window_size=10)
print("\nĐồ thị tổng IP Topo Đã Gắn Mask Không Rò Rỉ Thành Công!")

# tạo neighborloader (tương tự như dataloaders)
print("\nSet up IP Graph NeighborLoaders...")
train_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
test_loader_faiss  = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.test_mask,  batch_size=2048, shuffle=False, num_workers=0)

del df_all 
gc.collect()
print("Hoàn tất Data Prep cho IP Graph")

=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===
Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...
Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...
Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...
 - Tổng Node: 3,801,012
 - Train Mask : 2,470,638 (65.0%)
 - Valid Mask : 570,134 (15.0%)
 - Test Mask  : 760,240 (20.0%)
Hoàn tất Data Prep cho Masks và Sorting Thời gian.

=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/3801012 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
import gc

# định nghĩa gat với 2 lớp
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=7):
    from sklearn.metrics import f1_score
    import numpy as np

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3) 
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )
    
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    best_val_f1 = -1.0
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_nodes = 0
        all_train_preds = []
        all_train_labels = []
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
            batch_size = batch.batch_size
            
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            y_true = batch.y[:batch_size]
            
            all_train_preds.append(pred.cpu().numpy())
            all_train_labels.append(y_true.cpu().numpy())
            
            total_train_loss += loss.item() * batch_size
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
            
            del batch, out, emb, loss, pred, y_true
            
        avg_train_loss = total_train_loss / total_train_nodes
        train_f1 = f1_score(np.concatenate(all_train_labels), np.concatenate(all_train_preds), average='macro')
        
        pbar_train.close()
            
        model.eval()
        total_val_loss = 0
        total_val_nodes = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                
                out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                y_true = batch.y[:batch_size]
                
                all_val_preds.append(pred.cpu().numpy())
                all_val_labels.append(y_true.cpu().numpy())
                
                total_val_loss += loss.item() * batch_size
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}"})
                
                del batch, out, emb, loss, pred, y_true
                
        pbar_val.close()
                
        avg_val_loss = total_val_loss / total_val_nodes
        val_f1 = f1_score(np.concatenate(all_val_labels), np.concatenate(all_val_preds), average='macro')
        
        scheduler.step(val_f1)
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Macro F1: {train_f1:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Macro F1: {val_f1:.4f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate hiện tại: {current_lr:.6f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve_epochs = 0
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Macro F1: {best_val_f1:.4f} | Val Loss: {avg_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện F1 {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\nĐã kích hoạt Early Stopping tại Epoch {epoch+1}! Lùi về checkpoint tốt nhất.")
                break
                
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nHoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: {best_val_f1:.4f}) để dùng trích xuất Embedding!")
        
    return model

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm
import numpy as np
import torch
import gc
from torch_geometric.loader import NeighborLoader

# hàm trích xuất embedding từ model GAT
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader múc lấy Embeddings...")
    
    # loader dành cho lúc test/extract
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[20, 10], 
        input_nodes=mask,
        batch_size=512,
        shuffle=False,        
        num_workers=0 
    )
    
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        # truyền qua model để lấy embedding
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # lấy embedding của các node trong batch (chỉ lấy phần đầu tương ứng với batch_size vì NeighborLoader sẽ trả về cả node gốc và neighbor)
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        # xóa đi batch, emb để tránh nghẽn RAM
        del batch, emb, _
        
    pbar.close()
    
    # dọn rác cho các biến không dùng nữa
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # hợp final_embeddings thành ma trận (batch_size x embedding_dim) và final_labels thành vector (batch_size,)
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

# hàm huấn luyện và đánh giá XGBoost từ embedding của GAT
# train bằng tập valid, test trên tập test
def train_evaluate_xgboost(X_train_emb, y_train, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    # 1. Đã bỏ early_stopping_rounds
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, 
        learning_rate=0.08,
        max_depth=5,
        min_child_weight=2,
        gamma=1.0,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.75,         
        colsample_bytree=0.7,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda" 
    )
    
    print("Đang dọn dẹp RAM & VRAM...")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost (Chỉ trên tập Train)...")
    try:
        # 2. Đã bỏ eval_set
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            verbose=100
        )
        
    print(f"Huấn luyện xong XGBoost với tổng số cây: {xgb_model.n_estimators}")
    
    # Đánh giá trên tập Test
    print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [9]:
# load lại model để trích xuất embedding
num_features = super_graph_faiss.x.shape[1] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# lấy số classs
num_classes = len(torch.unique(super_graph_faiss.y)) 
model_faiss = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=128,      
     embedding_dim=64,        
     num_classes=num_classes, 
     heads=8,
     edge_dropout=0.3,
     edge_dim=4 
).to(device)
model_faiss.load_state_dict(torch.load(r"model_final\gat_embedder_24_4_best.pth", map_location=device))

C:\Users\Admin\AppData\Local\Temp\ipykernel_15368\2114185616.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_faiss.load_state_dict(torch.load(r"model_final\gat_em

<All keys matched successfully>

In [ ]:
# giải phóng RAM
import gc
gc.collect()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===")
# trích xuất sang Vector cho XGBoost xử lý
print("\nTrích xuất Vector cho Train Mask")
#X_train_faiss, y_train_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)

print("\nTrích xuất Vector cho Valid Mask")
X_valid_faiss, y_valid_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)

print("\nTrích xuất Vector cho Test Mask")
X_test_faiss, y_test_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

# huấn luyện và đánh giá XGBoost từ embedding của GAT (train bằng tập valid, test trên tập test)
hybrid_xgb_faiss = train_evaluate_xgboost(X_valid_faiss, y_valid_faiss, X_test_faiss, y_test_faiss)



=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===

Trích xuất Vector cho Train Mask

Trích xuất Vector cho Valid Mask
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1114 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (570134, 64)

Trích xuất Vector cho Test Mask
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1485 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (760240, 64)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
Đang dọn dẹp RAM & VRAM...
Đang huấn luyện XGBoost (Chỉ trên tập Train)...
Huấn luyện xong XGBoost với tổng số cây: 1000

--- ĐÁNH GIÁ TRÊN TẬP TEST ---
Accuracy:            99.11%
F1-Score (Macro):    96.72%
F1-Score (Weighted): 99.10%

Classification Report:
              precision    recall  f1-score   support

           0     0.9905    0.9753    0.9828     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.9996    1.0000    0.9998      2515
           3     1.0000    0.9870    0.9935     36253
           4     0.9167    0.8549    0.8847     18979
           5     1.0000    0.9992    0.9996      2451
           6     0.9732    0.8313    0.8967     11847
           7     1.0000    0.9941    0.9970    107436
           8     0.8710    0.9955    0.9291     16746
           9     0.9593    0.9